In [4]:
# ============================================================
# NOTEBOOK 12 — BATCH 3 PREPROCESSING
# CELL 1 — IMPORTS, PATHS, AND MODULE
# ============================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"D:\Pancreatic_Cancer_Thesis"
)

DATA_DIR = PROJECT_ROOT / "data"

RAW_CT_DIR = DATA_DIR / "raw_ct"

AUTO_LABEL_DIR = (
    DATA_DIR / "labels" / "Automatic_Labels"
)

MANUAL_LABEL_DIR = (
    DATA_DIR / "labels" / "Manual_Labels"
)

PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_IMAGES_DIR = (
    PROCESSED_DIR / "images"
)

PROCESSED_MASKS_DIR = (
    PROCESSED_DIR / "masks"
)

BATCH3_ELIGIBILITY_FILE = (
    PROCESSED_DIR / "batch3_eligibility.csv"
)

METADATA_FILE = (
    PROCESSED_DIR / "metadata.csv"
)

# ------------------------------------------------------------
# IMPORTANT:
# Add PROJECT_ROOT, not PROJECT_ROOT/src
#
# This allows:
#     from src import preprocessing as prep
#
# and lets preprocessing.py resolve:
#     from .io import ...
# ------------------------------------------------------------

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ------------------------------------------------------------
# Import project preprocessing module as a package
# ------------------------------------------------------------

from src import preprocessing as prep


print("=" * 70)
print("NOTEBOOK 12 — BATCH 3 PREPROCESSING")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nPreprocessing module:")
print(prep.__file__)

print("\nBatch 3 eligibility:")
print(BATCH3_ELIGIBILITY_FILE)

print("\nMetadata:")
print(METADATA_FILE)

print("\n✓ Project preprocessing module imported successfully.")

NOTEBOOK 12 — BATCH 3 PREPROCESSING

Project root:
D:\Pancreatic_Cancer_Thesis

Preprocessing module:
D:\Pancreatic_Cancer_Thesis\src\preprocessing.py

Batch 3 eligibility:
D:\Pancreatic_Cancer_Thesis\data\processed\batch3_eligibility.csv

Metadata:
D:\Pancreatic_Cancer_Thesis\data\processed\metadata.csv

✓ Project preprocessing module imported successfully.


In [5]:
# ============================================================
# CELL 2 — LOAD BATCH 3 ELIGIBILITY
# ============================================================

if not BATCH3_ELIGIBILITY_FILE.exists():
    raise FileNotFoundError(
        f"Eligibility file not found:\n"
        f"{BATCH3_ELIGIBILITY_FILE}"
    )

batch3_eligibility = pd.read_csv(
    BATCH3_ELIGIBILITY_FILE
)

print("=" * 70)
print("BATCH 3 ELIGIBILITY")
print("=" * 70)

print(
    "Total rows :",
    len(batch3_eligibility)
)

print(
    "Eligible   :",
    int(batch3_eligibility["eligible"].sum())
)

print(
    "Excluded   :",
    int((~batch3_eligibility["eligible"]).sum())
)

assert len(batch3_eligibility) == 580
assert batch3_eligibility["eligible"].sum() == 580
assert (~batch3_eligibility["eligible"]).sum() == 0

print("\n✓ All 580 Batch 3 cases are eligible.")

BATCH 3 ELIGIBILITY
Total rows : 580
Eligible   : 580
Excluded   : 0

✓ All 580 Batch 3 cases are eligible.


In [6]:
# ============================================================
# CELL 3 — BUILD BATCH 3 STUDY ID LIST
# ============================================================

batch3_eligible = batch3_eligibility[
    batch3_eligibility["eligible"]
].copy()

batch3_eligible_ids = (
    batch3_eligible["study_id"]
    .astype(str)
    .tolist()
)

print("=" * 70)
print("BATCH 3 STUDY IDS")
print("=" * 70)

print(
    "Eligible cases:",
    len(batch3_eligible_ids)
)

print("\nFirst 10:")
for study_id in batch3_eligible_ids[:10]:
    print(" ", study_id)

print("\nLast 10:")
for study_id in batch3_eligible_ids[-10:]:
    print(" ", study_id)

assert len(batch3_eligible_ids) == 580
assert len(set(batch3_eligible_ids)) == 580

print("\n✓ Study-ID list verified.")

BATCH 3 STUDY IDS
Eligible cases: 580

First 10:
  101112_00001
  101113_00001
  101114_00001
  101115_00001
  101116_00001
  101117_00001
  101118_00001
  101119_00001
  101120_00001
  101121_00001

Last 10:
  101681_00001
  101682_00001
  101683_00001
  101684_00001
  101685_00001
  101686_00001
  101687_00001
  101688_00001
  101689_00001
  101690_00001

✓ Study-ID list verified.


In [7]:
# ============================================================
# CELL 4 — CHECK EXISTING PROCESSED OUTPUTS
# ============================================================

PROCESSED_IMAGES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROCESSED_MASKS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

existing_images = {
    p.stem
    for p in PROCESSED_IMAGES_DIR.glob("*.npy")
}

existing_masks = {
    p.stem
    for p in PROCESSED_MASKS_DIR.glob("*.npy")
}

batch3_set = set(batch3_eligible_ids)

batch3_existing_images = (
    batch3_set & existing_images
)

batch3_existing_masks = (
    batch3_set & existing_masks
)

batch3_missing_images = (
    batch3_set - existing_images
)

batch3_missing_masks = (
    batch3_set - existing_masks
)

print("=" * 70)
print("EXISTING BATCH 3 OUTPUTS")
print("=" * 70)

print(
    "Batch 3 eligible cases :",
    len(batch3_set)
)

print(
    "Existing images       :",
    len(batch3_existing_images)
)

print(
    "Existing masks        :",
    len(batch3_existing_masks)
)

print(
    "Missing images        :",
    len(batch3_missing_images)
)

print(
    "Missing masks         :",
    len(batch3_missing_masks)
)

print("=" * 70)

EXISTING BATCH 3 OUTPUTS
Batch 3 eligible cases : 580
Existing images       : 0
Existing masks        : 0
Missing images        : 580
Missing masks         : 580


In [8]:
# ============================================================
# CELL 5 — VERIFY PREPROCESSING CONFIGURATION
# ============================================================

print("=" * 70)
print("PREPROCESSING CONFIGURATION")
print("=" * 70)

config_names = [
    "DEFAULT_TARGET_SPACING",
    "DEFAULT_ROI_SIZE",
    "DEFAULT_HU_WINDOW",
]

for name in config_names:

    if hasattr(prep, name):
        print(
            f"{name:<25}:",
            getattr(prep, name)
        )
    else:
        print(
            f"{name:<25}: NOT FOUND"
        )

print("\nExpected final ROI:")
print("  (128, 160, 192)")

PREPROCESSING CONFIGURATION
DEFAULT_TARGET_SPACING   : (1.0, 1.0, 3.0)
DEFAULT_ROI_SIZE         : (128, 160, 192)
DEFAULT_HU_WINDOW        : (-150, 250)

Expected final ROI:
  (128, 160, 192)


In [9]:
# ============================================================
# CELL 6 — PROCESS BATCH 3
# ============================================================

print("=" * 70)
print("STARTING BATCH 3 PREPROCESSING")
print("=" * 70)

print(
    "Requested cases:",
    len(batch3_eligible_ids)
)

print(
    "Overwrite:",
    False
)

print(
    "\nExisting outputs will be preserved."
)

batch3_metadata = prep.process_dataset(
    study_ids=batch3_eligible_ids,
    overwrite=False,
)

print("\n" + "=" * 70)
print("BATCH 3 PREPROCESSING COMPLETE")
print("=" * 70)

if batch3_metadata is not None:
    print(
        "Metadata rows returned:",
        len(batch3_metadata)
    )

STARTING BATCH 3 PREPROCESSING
Requested cases: 580
Overwrite: False

Existing outputs will be preserved.


Processing dataset:   0%|          | 0/580 [00:00<?, ?it/s]

After Resample   : -2048 3703
After Clip       : -150 250
After Normalize  : 0.0 1.0
After Crop       : 0.0 1.0
After Pad        : 0.0 1.0
Final            : 0.0 1.0
After Resample   : -2048 1690
After Clip       : -150 250
After Normalize  : 0.0 1.0
After Crop       : 0.0 1.0
After Pad        : 0.0 1.0
Final            : 0.0 1.0
After Resample   : -2048 1867
After Clip       : -150 250
After Normalize  : 0.0 1.0
After Crop       : 0.0 1.0
After Pad        : 0.0 1.0
Final            : 0.0 1.0
After Resample   : -1024 2792
After Clip       : -150 250
After Normalize  : 0.0 1.0
After Crop       : 0.0 1.0
After Pad        : 0.0 1.0
Final            : 0.0 1.0
After Resample   : -1024.0 2335.963
After Clip       : -150.0 250.0
After Normalize  : 0.0 1.0
After Crop       : 0.0 1.0
After Pad        : 0.0 1.0
Final            : 0.0 1.0
After Resample   : -1024 2265
After Clip       : -150 250
After Normalize  : 0.0 1.0
After Crop       : 0.0 1.0
After Pad        : 0.0 1.0
Final            : 0.

In [10]:
# ============================================================
# VERIFY BATCH 3 OUTPUT FILES
# ============================================================

print("=" * 70)
print("VERIFYING BATCH 3 OUTPUT FILES")
print("=" * 70)

image_files = {
    p.stem
    for p in PROCESSED_IMAGES_DIR.glob("*.npy")
}

mask_files = {
    p.stem
    for p in PROCESSED_MASKS_DIR.glob("*.npy")
}

batch3_ids = set(batch3_eligible_ids)

batch3_images = batch3_ids & image_files
batch3_masks = batch3_ids & mask_files

missing_images = batch3_ids - image_files
missing_masks = batch3_ids - mask_files

print("Total processed images :", len(image_files))
print("Total processed masks  :", len(mask_files))

print("\nBatch 3 expected       :", len(batch3_ids))
print("Batch 3 image outputs  :", len(batch3_images))
print("Batch 3 mask outputs   :", len(batch3_masks))
print("Missing images         :", len(missing_images))
print("Missing masks          :", len(missing_masks))

if missing_images:
    print("\nMissing images:")
    for study_id in sorted(missing_images):
        print(" ", study_id)

if missing_masks:
    print("\nMissing masks:")
    for study_id in sorted(missing_masks):
        print(" ", study_id)

if not missing_images and not missing_masks:
    print("\n✓ ALL 580 BATCH 3 CASES HAVE IMAGE + MASK OUTPUTS")
else:
    print("\n⚠ Batch 3 outputs are incomplete.")

VERIFYING BATCH 3 OUTPUT FILES
Total processed images : 1703
Total processed masks  : 1703

Batch 3 expected       : 580
Batch 3 image outputs  : 580
Batch 3 mask outputs   : 580
Missing images         : 0
Missing masks          : 0

✓ ALL 580 BATCH 3 CASES HAVE IMAGE + MASK OUTPUTS


In [11]:
# ============================================================
# FINAL FULL DATASET VERIFICATION
# ============================================================

print("=" * 70)
print("FINAL 1,703-CASE DATASET VERIFICATION")
print("=" * 70)

summary, report = prep.verify_dataset()

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

for key, value in summary.items():
    print(f"{key:<25}: {value}")

FINAL 1,703-CASE DATASET VERIFICATION


Verifying dataset:   0%|          | 0/1703 [00:00<?, ?it/s]

Processed Dataset Verification
num_cases                : 1703
missing_images           : 0
missing_masks            : 0
invalid_image_shape      : 0
invalid_mask_shape       : 0
invalid_image_dtype      : 0
invalid_mask_dtype       : 0
invalid_image_range      : 0
invalid_mask_labels      : 0
duplicate_study_ids      : 0

✓ All processed cases passed verification.

FINAL SUMMARY
num_cases                : 1703
missing_images           : 0
missing_masks            : 0
invalid_image_shape      : 0
invalid_mask_shape       : 0
invalid_image_dtype      : 0
invalid_mask_dtype       : 0
invalid_image_range      : 0
invalid_mask_labels      : 0
duplicate_study_ids      : 0


In [12]:
# ============================================================
# FINAL BATCH 3 RAW-CT DELETION SAFETY CHECK
# ============================================================

batch3_ids = set(batch3_eligible_ids)

kept_special_case = "100936_00001"

# Find the exact CT files that will be deleted
batch3_raw_files = [
    RAW_CT_DIR / f"{study_id}_0000.nii.gz"
    for study_id in batch3_ids
]

# Confirm special case is NOT in deletion list
assert kept_special_case not in batch3_ids

# Confirm every Batch 3 CT exists
missing = [
    str(path)
    for path in batch3_raw_files
    if not path.exists()
]

print("=" * 70)
print("BATCH 3 RAW CT DELETION SAFETY CHECK")
print("=" * 70)

print("Batch 3 cases       :", len(batch3_ids))
print("Raw CT files found  :", len(batch3_raw_files) - len(missing))
print("Missing raw CTs     :", len(missing))
print(
    "100936 CT preserved:",
    (
        RAW_CT_DIR
        / f"{kept_special_case}_0000.nii.gz"
    ).exists()
)

if missing:
    print("\nMissing files:")
    for path in missing:
        print(" ", path)
    raise RuntimeError(
        "Do not delete: some Batch 3 CT files are missing."
    )

print("\n✓ All 580 Batch 3 CTs are present.")
print("✓ 100936_00001 is excluded from deletion.")
print(
    "\nYou can now delete exactly these 580 files "
    "to make room for Batch 4."
)

BATCH 3 RAW CT DELETION SAFETY CHECK
Batch 3 cases       : 580
Raw CT files found  : 580
Missing raw CTs     : 0
100936 CT preserved: True

✓ All 580 Batch 3 CTs are present.
✓ 100936_00001 is excluded from deletion.

You can now delete exactly these 580 files to make room for Batch 4.
